In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run

/Workspace/Users/donprabhash0071@gmail.com/databricks/fmcg/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', '', 'Catalog')
dbutils.widgets.text('table', '', 'Table Name')

In [0]:
catalog = dbutils.widgets.get('catalog')
table = dbutils.widgets.get('table')

In [0]:
print(catalog, table)

In [0]:
df = spark.read.table(f'{catalog}.{bronze_schema}.{table}')
display(df.limit(10))

In [0]:
df = df.withColumn('customer_id', F.col('customer_id').cast('string'))

In [0]:
df.printSchema()

In [0]:
df = df.withColumn('customer_id', F.trim(F.col("customer_id")))
df = df.withColumn('customer_name', F.trim(F.col("customer_name")))
df = df.withColumn('city', F.trim(F.col("city")))

In [0]:
prev = df.count()
df = df.dropDuplicates()
print(f'Number of records dropped: {prev - df.count()}')

In [0]:
distinct_name = df.select('customer_name').distinct()
display(distinct_name)

In [0]:
df = df.withColumn('customer_name', F.initcap(F.col('customer_name')))

In [0]:
distinct_name = df.select('customer_name').distinct()
display(distinct_name)

In [0]:
distinct_city = df.select('city').distinct()
display(distinct_city)

#### Modify city names correctly.

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
newdelhi = 'NewDheli|NewDelhee|NewDelhee|NewDelhi'
hyd = 'Hyderabadd|Hyderbad'
blr = 'Bengalore|Bengaluruu'
df = df.withColumn('city', regexp_replace(F.col('city'), newdelhi, 'New Delhi'))
df = df.withColumn('city', regexp_replace(F.col('city'), hyd, 'Hyderabad'))
df = df.withColumn('city', regexp_replace(F.col('city'), blr, 'Bengaluru'))

In [0]:
distinct_city = df.select('city').distinct()
display(distinct_city)

In [0]:
display(df)

In [0]:
city_fix = {
    #Sprintx Nutrition
    789403: 'New Delhi',

    # Zenathlete Foods
    789420:'Bengaluru',

    # Primefuel Nutrition
    789521:'Hyderabad',
    
    # Recovery Lane
    789603: 'New Delhi',
     }

df_fix_city = spark.createDataFrame(
    [(k,v) for k,v in city_fix.items()],
    ['customer_id', 'fixed_city']
)

display(df_fix_city)

In [0]:
df = df.join(df_fix_city, 'customer_id', 'left')
display(df)

In [0]:
# replacing null cities with fixed city and then dropping fixed city
df = df.withColumn('city', F.coalesce('city', 'fixed_city')).drop('fixed_city')

In [0]:
display(df)

### Standardizing Column Names as per Gold Bucket

In [0]:
df = (
    # creating customer column
    df.withColumn('customer', F.concat_ws('-','customer_name', F.coalesce(F.col("city"), F.lit("unknown"))))

    # creating extra columns as per gold
    .withColumn('market', F.lit('India'))
    .withColumn('platform', F.lit('Sports Bar'))
    .withColumn('channel', F.lit('Acquisition'))
)

display(df.limit(5))

In [0]:
df.write\
    .format('Delta')\
    .mode('overwrite')\
    .option('delta.EnableChangeDataFeed', 'true')\
    .option('mergeSchema', 'true')\
    .saveAsTable(f'{catalog}.{silver_schema}.{table}')

### Gold Processing

In [0]:
df_silver = spark.sql(f'select * from {catalog}.{silver_schema}.{table}')
                      
# taking requested columns only
df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

In [0]:
df_gold.write\
    .mode('overwrite')\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{table}')


In [0]:
delta_table = DeltaTable.forName(spark, 'fmcg.gold.dim_customers')

In [0]:
df_child_customers = spark.read.table('fmcg.gold.sb_dim_customers').select(
    F.col('customer_id').alias('customer_code'),
    F.col('customer'),
    'market', 
    'platform', 
    'channel'
)

## Upsert Operation

In [0]:
delta_table.alias('target').merge(
    source = df_child_customers.alias('source'),
    condition = 'target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()